# 03 — Modelagem

**Objetivo:** comparar modelos, ajustar o melhor, avaliar no teste UMA vez. Versão final vive em `src/train.py`.

Roteiro:
1. Baseline: classe majoritária — F1 a superar
2. Validação cruzada (5 folds): LogisticRegression, RandomForest, GradientBoosting
3. GridSearchCV no melhor
4. Avaliação final no teste: classification_report, matriz de confusão, ROC-AUC
5. Interpretar: importância de features / coeficientes
6. **Copiar métricas pra `docs/technical.md` (seção 5) e README (tabela de resultados)**

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.train import build_pipeline, MODELS, TARGET

df = pd.read_csv("../data/processed/clean.csv")
X = df.drop(columns=[TARGET])
y = (df[TARGET].str.lower() == "yes").astype(int)
y.value_counts(normalize=True)

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

for name, model in MODELS.items():
    scores = cross_val_score(build_pipeline(model), X_train, y_train, cv=5, scoring="f1")
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

## Conclusões

Regressão Logística venceu (F1 CV 0.840 vs Gradient Boosting 0.823 vs Random Forest 0.802).

O modelo mais "simples" ganhou.
Dataset pequeno (~1250 linhas), features categóricas simples, relação quase linear entre elas e o alvo. Modelos complexos (florestas) brilham em padrões não-lineares — aqui não tem; só sobra risco de decorar ruído. Lição clássica: modelo mais complexo ≠ melhor.

4. Hiperparâmetros
GridSearch escolheu C = 0.1. Explicação simples: C controla o quanto o modelo pode "confiar" nos dados de treino. C baixo = mais regularização = coeficientes contidos = generaliza melhor em dataset pequeno. Testou 0.01, 0.1, 1, 10 — 0.1 venceu.

5. Métricas finais (teste, nunca visto no treino)
Métrica	Valor	Em português
F1 (yes)	0.86	equilíbrio precisão/recall (baseline era 0.672)
Recall (yes)	0.92	acha 92% de quem realmente busca tratamento
Precisão (yes)	0.81	quando diz "vai buscar", acerta 81%
Acurácia	0.85	acerta 85% do total
ROC-AUC	0.92	ótima separação entre classes
Matriz de confusão (251 pessoas no teste): 117 acertos "yes", 97 acertos "no", 27 falsos positivos, só 10 falsos negativos — bom, pois falso negativo (pessoa que precisaria de ajuda e passa despercebida) é o erro mais caro aqui.

O que o modelo aprendeu (coeficientes): quem diz que a condição mental interfere no trabalho "often" (+1.15) ou tem histórico familiar (+0.45) tende a buscar tratamento; quem responde "never" (-1.02) ou não informa (-1.82), não. Faz sentido psicológico — modelo aprendeu o esperado, sem maluquice.